In [77]:
"""ML"""
import torch
import torch.nn.functional as F
import numpy as np

"""Visualization"""
from matplotlib import pyplot as plt

"""Dataset and Preprocessing"""
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

"""Utils"""
# Add Base directory to Python path
import sys
import os
from pathlib import Path
sys.path.append(str(Path().resolve().parent))
from Utils.paths import *
import Utils.plotting as plotting
import Utils.json_inter as json_inter 
from Assets.Models.Models import *

from tqdm import tqdm

"""Logger"""
from Logger import *
logger = Logger(f"{LOGS}\\training.log", f"{LOGS}\\.sesh")

torch.manual_seed(0)
np.random.seed(0)

%load_ext autoreload
%autoreload 2

torch.__version__, np.__version__

Logger initialized with file D:\Repos\FPGA_Radar_Microdoppler_Classification\Logs\training.log, default tag [DEF] and level [INFO]
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


('2.5.1+cu118', '2.0.1')

In [78]:
json_inter.load_config_as_dataframe("preprocess_metadata.json")

,run_id,window_size,stride,snr_threshold,test_size,val_size,random_seed,balance_classes,dataset_name,file_name,timestamp,doppler_bins,correlation_threshold,max_lag,size,file_path
0,open_radar_K128_S32_SNR3.0_002,128,32,3.0,0.2,0.1,42,True,open_radar,open_radar_K128_S32_SNR3.npz,2026-02-23 13:45:53,,,,,
1,open_radar_K64_S32_SNR3.0_003,64,32,3.0,0.2,0.1,42,True,open_radar,open_radar_K64_S32_SNR3.npz,2026-02-23 13:54:47,,,,,
2,open_radar_K256_S32_SNR3.0_004,256,32,3.0,0.2,0.1,42,True,open_radar,open_radar_K256_S32_SNR3.npz,2026-02-23 13:58:14,1008.0,,,,
3,004,128,42,,0.1,0.1,42,,radar_windows_corr_stride,dataset,2026-02-24 19:50:38,1008.0,0.25,512.0,4.60 GB,
4,005,128,42,,0.1,0.1,0,,radar_windows_corr_stride,dataset,2026-02-24 20:17:44,1008.0,0.25,512.0,1.499 GB,D:\Repos\FPGA_Radar_Microdoppler_Classificatio...
5,006,64,42,,0.1,0.1,0,,radar_windows_corr_stride,dataset,2026-02-24 21:25:22,1008.0,0.25,512.0,0.8329 GB,D:\Repos\FPGA_Radar_Microdoppler_Classificatio...
6,007,32,32,,0.1,0.1,0,,radar_windows_corr_stride,dataset,2026-02-24 21:40:52,1008.0,0.25,512.0,0.576 GB,D:\Repos\FPGA_Radar_Microdoppler_Classificatio...
7,008,32,16,,0.1,0.1,0,,radar_windows_corr_stride,dataset,2026-02-24 21:44:38,1008.0,0.25,512.0,1.1336 GB,D:\Repos\FPGA_Radar_Microdoppler_Classificatio...
8,009,32,16,,0.1,0.1,0,,radar_windows_corr_stride,dataset,2026-02-24 22:41:20,256.0,0.25,512.0,0.1485 GB,D:\Repos\FPGA_Radar_Microdoppler_Classificatio...
9,010,32,16,,0.1,0.1,0,,radar_windows_corr_stride,dataset,2026-02-25 00:33:02,256.0,0.25,512.0,0.1485 GB,D:\Repos\FPGA_Radar_Microdoppler_Classificatio...


In [79]:
# # def load_processed_dataset(path,
# #                            to_torch=False,
# #                            device="cpu",
# #                            return_config=True,
# #                            verbose=True):
# #     """
# #     Loads radar window dataset and encodes string labels.
# #     """

# #     data = np.load(path, allow_pickle=True)

# #     train_X = data["train_X"]
# #     train_y = data["train_y"]
# #     val_X   = data["val_X"]
# #     val_y   = data["val_y"]
# #     test_X  = data["test_X"]
# #     test_y  = data["test_y"]

# #     config = data["config"].item() if "config" in data else None

# #     # -------------------------------------------------
# #     # 🔹 Encode string labels if necessary
# #     # -------------------------------------------------
# #     if train_y.dtype.type is np.str_ or train_y.dtype == object:

# #         all_labels = np.concatenate([train_y, val_y, test_y])
# #         classes = sorted(np.unique(all_labels))

# #         class_to_idx = {cls: i for i, cls in enumerate(classes)}
# #         idx_to_class = {i: cls for cls, i in class_to_idx.items()}

# #         train_y = np.array([class_to_idx[y] for y in train_y])
# #         val_y   = np.array([class_to_idx[y] for y in val_y])
# #         test_y  = np.array([class_to_idx[y] for y in test_y])

# #     else:
# #         class_to_idx = None
# #         idx_to_class = None

# #     # -------------------------------------------------
# #     # Convert to torch if requested
# #     # -------------------------------------------------
# #     if to_torch:
# #         train_X = torch.tensor(train_X, dtype=torch.float32, device=device)
# #         val_X   = torch.tensor(val_X, dtype=torch.float32, device=device)
# #         test_X  = torch.tensor(test_X, dtype=torch.float32, device=device)

# #         train_y = torch.tensor(train_y, dtype=torch.long, device=device)
# #         val_y   = torch.tensor(val_y, dtype=torch.long, device=device)
# #         test_y  = torch.tensor(test_y, dtype=torch.long, device=device)

# #     if verbose:
# #         print("="*50)
# #         print(f"Loaded: {path}")
# #         print("-"*50)
# #         print(f"Train: {train_X.shape}")
# #         print(f"Val  : {val_X.shape}")
# #         print(f"Test : {test_X.shape}")
# #         print(f"Num classes: {len(class_to_idx) if class_to_idx else 'Unknown'}")
# #         print("="*50)

# #     result = {
# #         "train_X": train_X,
# #         "train_y": train_y,
# #         "val_X": val_X,
# #         "val_y": val_y,
# #         "test_X": test_X,
# #         "test_y": test_y,
# #         "class_to_idx": class_to_idx,
# #         "idx_to_class": idx_to_class
# #     }

# #     if return_config:
# #         result["config"] = config

# #     return result
# # dataset_path = PATHS['PROCESSED_DATASETS'] +"\\"+ "open_radar_K64_S32_SNR3.0.npz"	
# # dataset = load_processed_dataset(
# #     dataset_path,
# #     to_torch=True,
# #     device="cuda"
# # )

# # train_X = dataset["train_X"]
# # train_y = dataset["train_y"]
# from collections import Counter

# def print_class_counts(dataset, name):
#     counts = Counter([item["label"] for item in dataset])
#     print(f"\n{name} distribution:")
#     for k, v in counts.items():
#         print(f"{k}: {v}")
#     print(f"Total: {sum(counts.values())}")

# print_class_counts(train_data, "Train (before balance)")
# print_class_counts(val_data, "Val (before balance)")
# print_class_counts(test_data, "Test (before balance)")

In [80]:
CONFIG = json_inter.read_json_config("model.config")

DATA_CONFIG = json_inter.get_config_by_id(LOGS+"\\"+"preprocess_metadata.json", CONFIG["dataset_id"])


CONFIG
DATA_CONFIG

CONFIG['dataset_path'] = DATA_CONFIG['file_path']
CONFIG['model_save_path'] = OUTPUTS + "\\" + "Models"
CONFIG, DATA_CONFIG

({'dataset_id': '012',
  'batch_size': 64,
  'epochs': 100,
  'learning_rate': 0.001,
  'use_class_weights': True,
  'model_name': 'cnn001',
  'dataset_path': 'D:\\Repos\\FPGA_Radar_Microdoppler_Classification\\Assets\\Dataset\\Processed\\radar_windows_corr_stride_K32_S16_DB512.npz',
  'model_save_path': 'D:\\Repos\\FPGA_Radar_Microdoppler_Classification\\Outputs\\Models'},
 {'window_size': 32,
  'stride': 16,
  'correlation_threshold': 0.25,
  'doppler_bins': 512,
  'max_lag': 512,
  'test_size': 0.1,
  'val_size': 0.1,
  'random_seed': 0,
  'dataset_name': 'radar_windows_corr_stride',
  'file_name': 'dataset',
  'file_path': 'D:\\Repos\\FPGA_Radar_Microdoppler_Classification\\Assets\\Dataset\\Processed\\radar_windows_corr_stride_K32_S16_DB512.npz',
  'timestamp': '2026-02-25 00:37:19',
  'size': '0.5766 GB'})

In [ ]:
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import os


class RadarDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]




def train_model(train_data, val_data, test_data):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_dataset = RadarDataset(*train_data)
    val_dataset   = RadarDataset(*val_data)
    test_dataset  = RadarDataset(*test_data)

    train_loader = DataLoader(train_dataset,
                              batch_size=CONFIG["batch_size"],
                              shuffle=True)

    val_loader = DataLoader(val_dataset,
                            batch_size=CONFIG["batch_size"],
                            shuffle=False)

    test_loader = DataLoader(test_dataset,
                             batch_size=CONFIG["batch_size"],
                             shuffle=False)

    input_height = train_data[0].shape[2]
    input_width  = train_data[0].shape[3]
    num_classes  = CONFIG["num_classes"]

    model = BasicRadarCNN(input_height, input_width, num_classes).to(device)

    # Optional class weighting
    if CONFIG.get("use_class_weights", False):
        class_counts = np.bincount(train_data[1])
        weights = 1.0 / (class_counts + 1e-8)
        weights = weights / weights.sum() * len(class_counts)
        weights = torch.tensor(weights, dtype=torch.float32).to(device)
        criterion = nn.CrossEntropyLoss(weight=weights)
    else:
        criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(model.parameters(), lr=CONFIG["learning_rate"])

    best_val_acc = 0.0

    logger("\nStarting Training...\n", tag=BEGIN)

    best_epoch = 0

    for epoch in range(CONFIG["epochs"]):

        model.train()
        train_loss = 0
        correct = 0
        total = 0

        for X, y in train_loader:
            X, y = X.to(device), y.to(device)

            optimizer.zero_grad()
            outputs = model(X)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = outputs.max(1)
            total += y.size(0)
            correct += predicted.eq(y).sum().item()

        train_acc = correct / total


        model.eval()
        val_loss = 0
        correct = 0
        total = 0

        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(device), y.to(device)
                outputs = model(X)
                loss = criterion(outputs, y)

                val_loss += loss.item()
                _, predicted = outputs.max(1)
                total += y.size(0)
                correct += predicted.eq(y).sum().item()

        val_acc = correct / total
        
        logger(f"Epoch [{epoch}/{CONFIG['epochs']}]")
        logger(f"Train ACC: {train_acc:.4f}")
        logger(f"Val ACC: {val_acc:.4f}")
        logger(f"Train Loss: {train_loss:.4f}")
        logger(f"Val Loss: {val_loss:.4f}")
        logger("="*30)
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch
            torch.save(model.state_dict(),
                       os.path.join(CONFIG["model_save_path"],
                                    CONFIG["model_name"] + ".pt"))

    logger("\n\nTraining Complete.")
    logger(f"Best Val Accuracy: {best_val_acc}")
    logger(f"Best Epoch: {best_epoch}")

    model.load_state_dict(torch.load(
        os.path.join(CONFIG["model_save_path"],
                     CONFIG["model_name"] + ".pt")
    ))

    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for X, y in test_loader:
            X, y = X.to(device), y.to(device)
            outputs = model(X)
            _, predicted = outputs.max(1)
            total += y.size(0)
            correct += predicted.eq(y).sum().item()

    test_acc = correct / total
    logger(f"Test Accuracy: {test_acc}")
    
    return model

In [82]:
logger.start_session()


logger("="*60)
logger("Loading dataset...")
logger(DATA_CONFIG.__str__(), tag=META)
logger(CONFIG.__str__(), tag=META)
logger("="*60)

dataset_path = CONFIG["dataset_path"]

data = np.load(dataset_path, allow_pickle=True)

train_X = data["train_X"]
train_y = data["train_y"]
val_X   = data["val_X"]
val_y   = data["val_y"]
test_X  = data["test_X"]
test_y  = data["test_y"]

logger(f"Train: {str(train_X.shape)}", tag=META)
logger(f"Val  : {str(val_X.shape)}", tag=META)
logger(f"Test : {str(test_X.shape)}", tag=META)


num_classes = len(np.unique(train_y))
CONFIG["num_classes"] = num_classes

logger(f"Num Classes: {num_classes}", tag=META)
logger(f"="*60)


os.makedirs(CONFIG["model_save_path"], exist_ok=True)


model = train_model(
    (train_X, train_y),
    (val_X, val_y),
    (test_X, test_y)
)


logger("\nExecution Complete.", tag=FINISH)


json_inter.save_config_to_json(CONFIG, "experiments_metadata")


logger.end_session()

[2026-02-25 00:38:06] [INFO] [MET]: START Session 11
[2026-02-25 00:38:06] [INFO] [DEF]: ============================================================
[2026-02-25 00:38:06] [INFO] [DEF]: Loading dataset...
[2026-02-25 00:38:06] [INFO] [MET]: {'window_size': 32, 'stride': 16, 'correlation_threshold': 0.25, 'doppler_bins': 512, 'max_lag': 512, 'test_size': 0.1, 'val_size': 0.1, 'random_seed': 0, 'dataset_name': 'radar_windows_corr_stride', 'file_name': 'dataset', 'file_path': 'D:\\Repos\\FPGA_Radar_Microdoppler_Classification\\Assets\\Dataset\\Processed\\radar_windows_corr_stride_K32_S16_DB512.npz', 'timestamp': '2026-02-25 00:37:19', 'size': '0.5766 GB'}
[2026-02-25 00:38:06] [INFO] [DEF]: ============================================================
[2026-02-25 00:38:09] [INFO] [MET]: Train: (8931, 1, 32, 512)
[2026-02-25 00:38:09] [INFO] [MET]: Val  : (995, 1, 32, 512)
[2026-02-25 00:38:09] [INFO] [MET]: Test : (1272, 1, 32, 512)
[2026-02-25 00:38:09] [INFO] [MET]: Num Classes: 4
[2026-

C:\Users\91748\AppData\Local\Temp\ipykernel_1436\1181449524.py:130: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(


[2026-02-25 00:44:57] [INFO] [DEF]: Test Accuracy: 0.9559748427672956
[2026-02-25 00:44:57] [INFO] [FIN]: Execution Complete.
[2026-02-25 00:44:57] [INFO] [MET]: END Session 11


In [83]:
json_inter.save_config_to_json(CONFIG, "experiments_metadata")


In [84]:
json_inter.load_config_as_dataframe("experiments_metadata")

,run_id,dataset_id,batch_size,epochs,learning_rate,use_class_weights,model_name,dataset_path,model_save_path,num_classes,timestamp,size
0,001,009,64,100,0.001,True,cnn001,D:\Repos\FPGA_Radar_Microdoppler_Classificatio...,D:\Repos\FPGA_Radar_Microdoppler_Classificatio...,4,2026-02-25 00:30:22,0.0 MB
1,002,011,64,100,0.001,True,cnn001,D:\Repos\FPGA_Radar_Microdoppler_Classificatio...,D:\Repos\FPGA_Radar_Microdoppler_Classificatio...,4,2026-02-25 00:35:57,0.0 MB
2,003,011,64,100,0.001,True,cnn001,D:\Repos\FPGA_Radar_Microdoppler_Classificatio...,D:\Repos\FPGA_Radar_Microdoppler_Classificatio...,4,2026-02-25 00:35:57,0.0 MB
3,004,012,64,100,0.001,True,cnn001,D:\Repos\FPGA_Radar_Microdoppler_Classificatio...,D:\Repos\FPGA_Radar_Microdoppler_Classificatio...,4,2026-02-25 00:44:57,0.0 MB
4,005,012,64,100,0.001,True,cnn001,D:\Repos\FPGA_Radar_Microdoppler_Classificatio...,D:\Repos\FPGA_Radar_Microdoppler_Classificatio...,4,2026-02-25 00:45:01,0.0 MB
